In [1]:
import os
import wandb # для логирования

import numpy as np
import random
from tqdm import *
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim # для оптимизаторов
from torchvision import datasets # для данных
import torchvision.transforms as transforms # для преобразований тензоров
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import joblib

import matplotlib.pyplot as plt

In [2]:
df= pd.read_csv('/Users/phuongnguyen/Downloads/car_policy.csv')

df.head()

,policy_tenure,age_of_car,age_of_policyholder,population_density,make,max_torque,max_power,airbags,is_esc,is_adjustable_steering,...,engine_type_K Series Dual jet,engine_type_K10C,engine_type_i-DTEC,rear_brakes_type_Drum,transmission_type_Manual,steering_type_Manual,steering_type_Power,safe_score,car_size,age_of_car_and_policy
0,0.515874,0.05,0.644231,4990,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.025794
1,0.672619,0.02,0.375000,27003,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.013452
2,0.841110,0.02,0.384615,4076,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.016822
3,0.900277,0.11,0.432692,21622,1,113.0,88.50,2,1,1,...,0,0,0,1,0,0,0,6,10500957375,0.099030
4,0.596403,0.11,0.634615,34738,2,91.0,67.06,2,0,0,...,0,0,0,1,0,0,0,3,8777961010,0.065604


In [3]:
# Разделение на X и y
X = df.drop(columns = ['is_claim'])
y = df['is_claim']

print(X.shape)
print(y.shape)

(58592, 89)
(58592,)


In [4]:
y.value_counts()

is_claim
0    54844
1     3748
Name: count, dtype: int64

In [5]:
# Зафиксируем seed для воспроизводимости

def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    #torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    #torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток

In [6]:
# функция перевода класса конфигурации в словарь

def class2dict(f):
  return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

In [ ]:
class CFG:

# Задаем параметры нашего эксперимента

  api = "---------------------"# вписать свой API Wandb
  project = "Models"# вписать название эксперимента, который предварительно надо создать в Wandb
  num_epochs = 15 # количество эпох
  train_batch_size = 64 # размер батча обучающей выборки
  test_batch_size = 512 # размер батча тестовой выборки
  num_workers = 2 # количество активных процессов на загрузку данных
  lr = 0.001 # learning_rate
  seed = 2022 # для функции воспроизводимости
  wandb = True # флаг использования Wandb

In [8]:
 #Поделим данные на train, test

X_train, X_test,  y_train, y_test  = train_test_split(X, y, test_size= 0.2, random_state = CFG.seed, stratify = y)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(46873, 89)
(11719, 89)
(46873,)
(11719,)


In [9]:
# Стандартизируем наги значения 
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Превращаю в тензор
X_train_tens = torch.tensor(X_train_scaled, dtype  = torch.float32)
X_test_tens = torch.tensor(X_test_scaled, dtype  = torch.float32)
y_train_tens = torch.tensor(y_train.values, dtype  = torch.float32).reshape(-1, 1)
y_test_tens = torch.tensor(y_test.values, dtype  = torch.float32).reshape(-1, 1)


# Создаю train_dataset из X_train_tens и y_train_tens и  test_dataset из X_test_tens и y_test_tens
train_dataset = TensorDataset(X_train_tens, y_train_tens)
test_dataset = TensorDataset(X_test_tens, y_test_tens)

#создаю лоудары, чтобы передавались данные батчами
train_loader = DataLoader(train_dataset, batch_size= CFG.train_batch_size, shuffle= True, num_workers= CFG.num_workers)
test_loader = DataLoader(test_dataset, batch_size= CFG.test_batch_size, shuffle = False, num_workers= CFG.num_workers)




In [10]:
examples = enumerate(train_loader)

batch_ind, (example_data, example_targets ) = next(examples)

In [11]:
example_data.shape

torch.Size([64, 89])

In [12]:
example_targets.shape

torch.Size([64, 1])

## Модель 1 

In [13]:


class Model_1 (nn.Module):
    
    def __init__(self):
        super(Model_1,self).__init__()
        
        hidden_1 = 256
        hidden_2 = 128
        hidden_3 = 64
        
        # первый слой (89 -> hidden_1)
        self.fc1 = nn.Linear(89, hidden_1)
        self.batch_norm1 = nn.BatchNorm1d(hidden_1)
        self.act1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)    
    
        # второй слой (hidden_1 -> hidden_2)
        self.fc2  = nn.Linear(hidden_1, hidden_2)
        self.batch_norm2 = nn.BatchNorm1d(hidden_2)
        self.act2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)  
        
        # третий слой (hidden_2 -> hidden_3)
        self.fc3  = nn.Linear(hidden_2, hidden_3)
        self.batch_norm3 = nn.BatchNorm1d(hidden_3)
        self.act3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.2)
        
        #вывходной слой
        self.fc4 = nn.Linear(hidden_3, 1)
        
    def forward(self, x):
        
            
        x = self.fc1(x)
        x = self.batch_norm1(x)
        x = self.act1(x)
        x = self.dropout1(x)
            
            
        x = self.fc2(x)
        x = self.batch_norm2(x)
        x = self.act2(x)
        x = self.dropout2(x)
            
        x = self.fc3(x)
        x = self.batch_norm3(x)
        x = self.act3(x)
        x = self.dropout3(x)
            
        x = self.fc4(x)
            
        return x

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Model_1().to(device)

print(device)
print(model)

cpu
Model_1(
  (fc1): Linear(in_features=89, out_features=256, bias=True)
  (batch_norm1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): ReLU()
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (batch_norm2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (batch_norm3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act3): ReLU()
  (dropout3): Dropout(p=0.2, inplace=False)
  (fc4): Linear(in_features=64, out_features=1, bias=True)
)


In [15]:
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32).to(device) #говорим что ошибка на классе 1 важнее, чем ошибка на классе 0
# функция потерь 
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

#оптимизатор
optimizer = torch.optim.Adam(model.parameters(), lr = CFG.lr) #https://docs.pytorch.org/docs/main/generated/torch.optim.Adam.html

In [16]:

#крч проверяю на одном батче как проходит и считает loss
example_data = example_data.to(device)
example_targets = example_targets.to(device)

outputs = model(example_data)

loss = criterion(outputs, example_targets)

print(outputs.shape)
print(example_targets.shape)
print(loss.item())

torch.Size([64, 1])
torch.Size([64, 1])
1.1074912548065186


In [17]:



# функция обучения модели
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score #https://scikit-learn.ru/stable/modules/model_evaluation.html

def train(model, device, train_loader, optimizer, criterion, epoch, WANDB):
    model.train()
    
    
    train_loss = 0
    correct = 0
    total = 0
    n_ex = len(train_loader)
    all_preds = []
    all_targets = []
    
    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total = n_ex):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        #прямой проход
        output = model(data)
        loss = criterion(output, target)
        train_loss += loss.item()
        probs = torch.sigmoid(output) #считаю вероятность https://docs.pytorch.org/docs/main/generated/torch.nn.Sigmoid.html
        pred = (probs >= 0.55).float() #считаю классы 0/1, если вероятность >= 0.5, ставим класс 1
        correct += pred.eq(target).sum().item()
        total += target.size(0)
        #собраю все предсказания и все реальные ответы со всех батчей в обычные списки, чтобы потом посчитать precision, recall, f1
        all_preds.extend(pred.detach().cpu().numpy().ravel()) #https://docs.pytorch.org/docs/2.12/generated/torch.Tensor.detach.html
        all_targets.extend(target.detach().cpu().numpy().ravel()) 
        #обратный проход
        loss.backward()
        #градиентный шаг
        optimizer.step()
        
    #считаю метрики
    train_loss = train_loss / len(train_loader)
    train_accuracy = correct / total
    train_precision = precision_score(all_targets, all_preds, zero_division = 0)
    train_recall = recall_score(all_targets, all_preds, zero_division = 0)
    train_f1 = f1_score(all_targets, all_preds, zero_division = 0)
    
    tqdm.write('\nTrain Epoch: {} | Average loss: {:.4f} | Accuracy: {:.2f}% | Precision: {:.4f} | Recall: {:.4f} | F1: {:.4f}'.format(epoch,train_loss,100. * train_accuracy,train_precision, train_recall, train_f1))
    
        
    if WANDB:
            wandb.log({ 'epoch': epoch, 'train_loss': train_loss,'train_accuracy': train_accuracy,'train_precision': train_precision,'train_recall': train_recall,'train_f1': train_f1})
        
        
    return train_loss, train_accuracy, train_precision, train_recall, train_f1
        

In [18]:
#фунция тестирования 
def test(model, device, test_loader, criterion, WANDB=False):
    model.eval()

    test_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            data = data.to(device)
            target = target.to(device).float()
            output = model(data)
            loss = criterion(output, target)
            test_loss += loss.item()
            probs = torch.sigmoid(output)
            pred = (probs >= 0.55).float()
            correct += pred.eq(target).sum().item()
            total += target.size(0)
            all_preds.extend(pred.detach().cpu().numpy().ravel())
            all_targets.extend(target.detach().cpu().numpy().ravel())

    test_loss = test_loss / len(test_loader)
    test_accuracy = correct / total

    test_precision = precision_score(all_targets, all_preds, zero_division = 0)
    test_recall = recall_score(all_targets, all_preds, zero_division = 0)
    test_f1 = f1_score(all_targets, all_preds, zero_division = 0)

    tqdm.write('\nTest set: Average loss: {:.4f} | Accuracy: {:.2f}% | Precision: {:.4f} | Recall: {:.4f} | F1: {:.4f}'.format(test_loss,100. * test_accuracy, test_precision, test_recall, test_f1 ) )

    if WANDB:
        wandb.log({'test_loss': test_loss,'test_accuracy': test_accuracy,'test_precision': test_precision, 'test_recall': test_recall, 'test_f1': test_f1})
        
    return test_loss, test_accuracy, test_precision, test_recall, test_f1
 
 

    

In [ ]:
#основная функция для эксперимента
def run_experiment(model, model_name):
    
    seed_everything(CFG.seed)
  
    use_cuda= torch.cuda.is_available() #Проверем доступность gpu
    device = torch.device("cuda" if use_cuda else 'cpu') #выделили устройство
    model = model.to(device)
    

    
    pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype = torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight = pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr = CFG.lr)



    
    if CFG.wandb:
        wandb.init(
            project=CFG.project,
            name=model_name,
            config={ **class2dict(CFG), "model_name": model_name, 'architecture' : str(model), 'epochs': CFG.num_epochs, 'batch_size': CFG.train_batch_size, 'lr': CFG.lr, 'optimizer': 'Adam',  'loss': 'BCEWithLogitsLoss', 'pos_weight': pos_weight.item(), 'threshold': 0.5, 'seed': CFG.seed})



    for epoch in range(1, CFG.num_epochs + 1):
        train(model,device,train_loader, optimizer, criterion,epoch, WANDB = CFG.wandb )
        test(model, device, test_loader, criterion,WANDB  = CFG.wandb)
    torch.save(model.state_dict(), f'{model_name}.pth')
    
    joblib.dump(scaler, 'scaler.pkl') #сохраняем скаллер

    #сохраняем данные 
    X_train.to_csv('X_train.csv', index = False)
    X_test.to_csv('X_test.csv', index = False)
    y_train.to_csv('y_train.csv', index = False)
    y_test.to_csv('y_test.csv', index = False)

    if CFG.wandb:
        
        artifact = wandb.Artifact(name=f'{model_name}_artifacts', type = 'model') # работа с артефактами -  https://docs.wandb.ai/models/ref/python/experiments и https://wandb.ai/wandb/common-ml-errors/reports/How-to-save-and-load-models-in-PyTorch--VmlldzozMjg0MTE

        #добавляю эти файлы в W&B artifact
        artifact.add_file(f'{model_name}.pth')
        artifact.add_file('scaler.pkl')
        artifact.add_file('X_train.csv')
        artifact.add_file('X_test.csv')
        artifact.add_file('y_train.csv')
        artifact.add_file('y_test.csv')
        wandb.log_artifact(artifact)
        wandb.finish()
        
        
        
        

In [20]:
CFG.wandb

True

In [21]:
run_experiment(model, 'model_1')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/phuongnguyen/.netrc.
wandb: Currently logged in as: huesospro2005 (huesospro2005-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 733/733 [00:02<00:00, 328.46it/s]


Train Epoch: 1 | Average loss: 1.3058 | Accuracy: 73.13% | Precision: 0.0776 | Recall: 0.2942 | F1: 0.1228



Test set: Average loss: 1.2649 | Accuracy: 79.14% | Precision: 0.0997 | Recall: 0.2813 | F1: 0.1472


100%|██████████| 733/733 [00:02<00:00, 297.07it/s]

...


Test set: Average loss: 1.2396 | Accuracy: 61.38% | Precision: 0.0954 | Recall: 0.5933 | F1: 0.1643


100%|██████████| 733/733 [00:02<00:00, 343.58it/s]


Train Epoch: 15 | Average loss: 1.2513 | Accuracy: 66.61% | Precision: 0.0926 | Recall: 0.4797 | F1: 0.1552



Test set: Average loss: 1.2468 | Accuracy: 71.52% | Precision: 0.0992 | Recall: 0.4267 | F1: 0.1609


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,██▃▇▅▆▃▅▆▅▅▄▇▁▅
test_f1,▁▂▄▄▅█▇▇█▆▆▆▇█▇
test_loss,██▇▅▆▆▄▄▄▄▄▂▄▁▃
test_precision,▅▇▁▇▄█▄▅▇▄▅▄█▃▅
test_recall,▁▁▆▂▄▃▆▄▄▅▄▅▂█▄
train_accuracy,█▇▆▆▆▆▆▄▄▄▄▄▁▄▁
train_f1,▁▄▅▆▅▆▇▆█▇█▇▇██
train_loss,█▅▃▃▃▃▂▂▂▂▂▁▁▁▁
train_precision,▁▄▅▆▆▇▇▆█▇█▇▇█▇
+1,...


Первая модель использовалась как базовая регуляризованная MLP для табличной бинарной классификации. Архитектура 89–256–128–64–1 выбрана как умеренно сложная: она достаточно глубокая, чтобы выявлять нелинейные зависимости между признаками автомобиля и страхового полиса, но при этом не слишком большая, чтобы резко увеличить риск переобучения. После скрытых слоёв применяются BatchNorm1d и Dropout: BatchNorm1d стабилизирует обучение, а Dropout снижает вероятность переобучения. В качестве функции активации используется ReLU, так как она стандартно применяется в полносвязных нейронных сетях и хорошо работает с глубокими архитектурами. Для обучения используется BCEWithLogitsLoss с pos_weight, потому что целевой класс 1 встречается значительно реже класса 0, и без учёта дисбаланса модель склонна предсказывать только класс 0

### Воспроизведение без обучения

In [22]:
# Загружаем данные
X_test = pd.read_csv('/Users/phuongnguyen/Downloads/X_test.csv')
y_test = pd.read_csv('/Users/phuongnguyen/Downloads/y_test.csv')

scaler = joblib.load('/Users/phuongnguyen/Downloads/scaler.pkl') #загружаю скаллер

X_test_scaled = scaler.transform(X_test) #стандартизирую как при обучении




X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32) #перевод в тензор

model = Model_1()

model.load_state_dict(torch.load('/Users/phuongnguyen/Downloads/model_1 (1).pth', map_location = 'cpu')) #загружаем сохранённые веса обратно в модель

model.eval()

with torch.no_grad(): #Получаем предсказания без обучения
    output = model(X_test_tensor)
    probs = torch.sigmoid(output)
    preds = (probs >= 0.5).float()
    
print(output.shape)
print(probs[:10])
print(preds[:10])
print(preds.sum())


y_true = y_test.values.ravel()
y_pred = preds.numpy().ravel()

#проверяю качество без обучения
print('accuracy:', accuracy_score(y_true, y_pred))
print('precision:', precision_score(y_true, y_pred, zero_division = 0))
print('recall:', recall_score(y_true, y_pred, zero_division = 0))
print('f1:', f1_score(y_true, y_pred, zero_division = 0))





torch.Size([11719, 1])
tensor([[0.5496],
        [0.4605],
        [0.5880],
        [0.4220],
        [0.3971],
        [0.4382],
        [0.5600],
        [0.4682],
        [0.5003],
        [0.4275]])
tensor([[1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.]])
tensor(5021.)
accuracy: 0.5852035156583326
precision: 0.09061939852619
recall: 0.6066666666666667
f1: 0.1576849766071738


модель находит много реальных страховых случаев, потому что recall ≈ 60.7%, но делает много ложных тревог, потому что precision ≈ 9.1%

## Модель 2

Вторая модель отличается от первой увеличенной глубиной и шириной сети: вместо архитектуры 89–256–128–64–1 используется 89–512–256–128–64–32–1. Это позволяет проверить, улучшит ли более сложная MLP способность модели выявлять нелинейные зависимости между признаками страхового полиса и фактом наступления страхового случая. Для контроля переобучения сохранены BatchNorm1d и Dropout, причём dropout в первых слоях увеличен

In [35]:


class Model_2 (nn.Module):
    
    def __init__(self):
        super(Model_2,self).__init__()
        
        hidden_1 = 512
        hidden_2 = 256
        hidden_3 = 128
        hidden_4 = 64
        hidden_5 = 32
        
        # первый слой (89 -> hidden_1)
        self.fc1 = nn.Linear(89, hidden_1)
        self.batch_norm1 = nn.BatchNorm1d(512)
        self.act1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.4)
        
        #второй слой (hidden_1 -> hidden_2)
        self.fc2 = nn.Linear(hidden_1, hidden_2)
        self.batch_norm2 = nn.BatchNorm1d(256)
        self.act2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        
        #третий слой (hidden_2 -> hidden_3)      
        self.fc3 = nn.Linear(hidden_2, hidden_3)
        self.batch_norm3 = nn.BatchNorm1d(128)
        self.act3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.3) 
        
        #четвертый слой (hidden_3 -> hidden_4)           
        self.fc4 = nn.Linear(hidden_3, hidden_4)
        self.batch_norm4 = nn.BatchNorm1d(64)
        self.act4 = nn.ReLU()
        self.dropout4 = nn.Dropout(0.2) 
        
        #пятый слой (hidden_4 -> hidden_5)           
        self.fc5 = nn.Linear(hidden_4, hidden_5)
        self.batch_norm5 = nn.BatchNorm1d(32)
        self.act5 = nn.ReLU()
        self.dropout5 = nn.Dropout(0.1) 
        
        #Выходной слой
        self.fc6 = nn.Linear(hidden_5, 1)
        
    def forward(self, x):
            
        x = self.fc1(x)
        x = self.batch_norm1(x)
        x = self.act1(x)
        x = self.dropout1(x)
                
                
        x = self.fc2(x)
        x = self.batch_norm2(x)
        x = self.act2(x)
        x = self.dropout2(x)
                
        x = self.fc3(x)
        x = self.batch_norm3(x)
        x = self.act3(x)
        x = self.dropout3(x)
                
        x = self.fc4(x)
        x = self.batch_norm4(x)
        x = self.act4(x)
        x = self.dropout4(x)
             
        x = self.fc5(x)
        x = self.batch_norm5(x)
        x = self.act5(x)
        x = self.dropout5(x)
            
        x = self.fc6(x)

        return x

In [34]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_2 = Model_2().to(device)

print(device)
print(model_2)

cpu
Model_2(
  (fc1): Linear(in_features=89, out_features=512, bias=True)
  (batch_norm1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): ReLU()
  (dropout1): Dropout(p=0.4, inplace=False)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (batch_norm2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (batch_norm3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act3): ReLU()
  (dropout3): Dropout(p=0.3, inplace=False)
  (fc4): Linear(in_features=128, out_features=64, bias=True)
  (batch_norm4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act4): ReLU()
  (dropout4): Dropout(p=0.3, inplace=False)
  (fc5): Linear(in_features=64, out_features=32, bias=True)
  

In [31]:
run_experiment(model_2, 'model_2')

100%|██████████| 733/733 [00:03<00:00, 216.00it/s]



Train Epoch: 1 | Average loss: 1.3059 | Accuracy: 73.28% | Precision: 0.0728 | Recall: 0.2708 | F1: 0.1148

Test set: Average loss: 1.2717 | Accuracy: 75.01% | Precision: 0.1023 | Recall: 0.3733 | F1: 0.1606


100%|██████████| 733/733 [00:03<00:00, 219.86it/s]


Train Epoch: 2 | Average loss: 1.2825 | Accuracy: 74.75% | Precision: 0.0805 | Recall: 0.2829 | F1: 0.1253



Test set: Average loss: 1.2709 | Accuracy: 58.00% | Precision: 0.0848 | Recall: 0.5680 | F1: 0.1476


...


Train Epoch: 14 | Average loss: 1.2536 | Accuracy: 65.78% | Precision: 0.0916 | Recall: 0.4877 | F1: 0.1542



Test set: Average loss: 1.2466 | Accuracy: 67.49% | Precision: 0.0931 | Recall: 0.4667 | F1: 0.1552


100%|██████████| 733/733 [00:03<00:00, 220.47it/s]



Train Epoch: 15 | Average loss: 1.2532 | Accuracy: 64.56% | Precision: 0.0914 | Recall: 0.5077 | F1: 0.1548

Test set: Average loss: 1.2489 | Accuracy: 64.32% | Precision: 0.0934 | Recall: 0.5253 | F1: 0.1586


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,▅▂█▅▃▅█▃▄▃▁▄▃▄▃
test_f1,█▇▁█▇▆▅██▇▇██▇█
test_loss,██▆▃▄▄▅▂▂▂▂▃▁▁▂
test_precision,▄▁█▄▂▃█▃▃▂▂▃▃▃▃
test_recall,▅▇▁▅▆▄▂▆▆▆█▆▇▆▆
train_accuracy,▇█▅▅▆▆▄▆▃▃▂▃▄▂▁
train_f1,▁▃▆▆▆▆▇▆▇█▇▇▇██
train_loss,█▅▄▃▃▃▂▂▂▂▂▁▁▁▁
train_precision,▁▄▆▆▇▇▇▇▇█▇▇███
+1,...


Вторая модель была построена как более глубокая и широкая версия базовой MLP. По сравнению с первой конфигурацией архитектура была увеличена с 89–256–128–64–1 до 89–512–256–128–64–32–1. Основная цель этой конфигурации это проверить, сможет ли более сложная нейронная сеть лучше выявлять нелинейные зависимости между характеристиками автомобиля, страхового полиса и фактом наступления страхового случая. Функция активации ReLU, оптимизатор Adam, learning rate, batch size и количество эпох были оставлены такими же, как в первой модели, чтобы сравнение отражало именно влияние изменения архитектуры, а не изменение условий обучения для контроля переобучения использовались BatchNorm1d и Dropout, причём в первом слое dropout был увеличен до 0.4, так как модель стала более сложной и имеет больше обучаемых параметров

Вторая модель стала лучше находить класс 1, потому что recall вырос с 0.4267 до 0.5253. То есть она чаще ловит реальные страховые случаи, но при этом precision и accuracy стали ниже, а F1 чуть хуже: 0.1586 против 0.1609. Значит более глубокая и широкая архитектура не дала общего улучшения качества, а просто сместила модель в сторону более агрессивного предсказания класса 1